# Phase 1 - Data Audit

Notebook này kiểm tra chất lượng dữ liệu ban đầu cho project **E-commerce Sales & Customer Behavior Analytics**.

Mục tiêu:

- Hiểu cấu trúc raw dataset.
- Kiểm tra shape, schema, data types.
- Kiểm tra missing values và duplicate records.
- Kiểm tra numeric metrics như `Sales`, `Quantity`, `Discount`, `Profit`, `Shipping_Cost`.
- Kiểm tra categorical values như `Product_Category`, `Payment_method`, `Device_Type`.
- Export audit tables để dùng trong report và portfolio.

## 1. Import Libraries

In [ ]:
from pathlib import Path
from typing import Iterable

import pandas as pd

## 2. Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "E-commerce Dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "data_audit_tables"

REQUIRED_COLUMNS = [
    "Order_Date",
    "Time",
    "Aging",
    "Customer_Id",
    "Gender",
    "Device_Type",
    "Customer_Login_type",
    "Product_Category",
    "Product",
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "Shipping_Cost",
    "Order_Priority",
    "Payment_method",
]

NUMERIC_COLUMNS = [
    "Aging",
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "Shipping_Cost",
]

CATEGORICAL_COLUMNS = [
    "Gender",
    "Device_Type",
    "Customer_Login_type",
    "Product_Category",
    "Order_Priority",
    "Payment_method",
]

## 3. Helper Functions

In [ ]:
def validate_columns(df: pd.DataFrame, required_columns: Iterable[str]) -> None:
    """Validate that the raw dataset contains all expected columns."""
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")


def load_raw_data(file_path: Path) -> pd.DataFrame:
    """Load raw CSV data and perform minimal schema validation."""
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset not found: {file_path}")

    df = pd.read_csv(file_path)
    validate_columns(df, REQUIRED_COLUMNS)
    return df


def prepare_audit_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Convert audit-critical fields to the right data types without dropping rows."""
    audit_df = df.copy()
    audit_df["Order_Date"] = pd.to_datetime(audit_df["Order_Date"], errors="coerce")

    for col in NUMERIC_COLUMNS:
        audit_df[col] = pd.to_numeric(audit_df[col], errors="coerce")

    return audit_df

## 4. Load Dataset

In [ ]:
try:
    raw_df = load_raw_data(DATA_PATH)
    audit_df = prepare_audit_frame(raw_df)
    print(f"Dataset loaded successfully: {audit_df.shape[0]:,} rows x {audit_df.shape[1]} columns")
except Exception as error:
    print(f"Failed to load dataset: {error}")
    raise

In [ ]:
audit_df.head(10)

## 5. Dataset Shape and Schema

In [ ]:
shape_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns", "duplicate_rows", "unique_customers", "unique_products"],
        "value": [
            len(audit_df),
            audit_df.shape[1],
            audit_df.duplicated().sum(),
            audit_df["Customer_Id"].nunique(),
            audit_df["Product"].nunique(),
        ],
    }
)

shape_summary

In [ ]:
schema_summary = pd.DataFrame(
    {
        "column_name": audit_df.columns,
        "data_type": audit_df.dtypes.astype(str).values,
        "non_null_count": audit_df.notna().sum().values,
        "unique_count": audit_df.nunique(dropna=True).values,
    }
)

schema_summary

## 6. Date Coverage

In [ ]:
date_summary = pd.DataFrame(
    {
        "metric": ["min_order_date", "max_order_date", "invalid_order_dates"],
        "value": [
            audit_df["Order_Date"].min(),
            audit_df["Order_Date"].max(),
            audit_df["Order_Date"].isna().sum(),
        ],
    }
)

date_summary

## 7. Missing Values

In [ ]:
missing_values = (
    pd.DataFrame(
        {
            "column_name": audit_df.columns,
            "missing_count": audit_df.isna().sum().values,
            "missing_percent": (audit_df.isna().sum().values / len(audit_df) * 100).round(4),
        }
    )
    .query("missing_count > 0")
    .sort_values(["missing_count", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

missing_values

In [ ]:
missing_examples = audit_df.loc[audit_df.isna().any(axis=1)].copy()
missing_examples.insert(0, "csv_line_number", missing_examples.index + 2)
missing_examples["missing_columns"] = missing_examples.apply(
    lambda row: ", ".join(row.index[row.isna()].tolist()),
    axis=1,
)

missing_examples

## 8. Numeric Summary

In [ ]:
numeric_summary = audit_df[NUMERIC_COLUMNS].describe().T.reset_index()
numeric_summary = numeric_summary.rename(columns={"index": "column_name"})
numeric_summary["missing_count"] = audit_df[NUMERIC_COLUMNS].isna().sum().values
numeric_summary["sum"] = audit_df[NUMERIC_COLUMNS].sum(numeric_only=True).values

numeric_summary = numeric_summary[
    ["column_name", "count", "missing_count", "mean", "std", "min", "25%", "50%", "75%", "max", "sum"]
].round(4)

numeric_summary

## 9. Categorical Distribution

In [ ]:
categorical_tables = []

for col in CATEGORICAL_COLUMNS:
    distribution = (
        audit_df[col]
        .fillna("<missing>")
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={col: "value", "count": "row_count"})
    )
    distribution.insert(0, "column_name", col)
    distribution["row_percent"] = (distribution["row_count"] / len(audit_df) * 100).round(4)
    categorical_tables.append(distribution)

categorical_distribution = pd.concat(categorical_tables, ignore_index=True)
categorical_distribution

## 10. Category KPI Summary

In [ ]:
category_kpi_summary = (
    audit_df.groupby("Product_Category", dropna=False)
    .agg(
        orders=("Product", "count"),
        customers=("Customer_Id", "nunique"),
        products=("Product", "nunique"),
        revenue=("Sales", "sum"),
        quantity=("Quantity", "sum"),
        profit=("Profit", "sum"),
        shipping_cost=("Shipping_Cost", "sum"),
        avg_discount=("Discount", "mean"),
    )
    .reset_index()
)

category_kpi_summary["profit_margin"] = category_kpi_summary["profit"] / category_kpi_summary["revenue"]
category_kpi_summary["shipping_cost_ratio"] = category_kpi_summary["shipping_cost"] / category_kpi_summary["revenue"]
category_kpi_summary["revenue_share"] = category_kpi_summary["revenue"] / category_kpi_summary["revenue"].sum()

category_kpi_summary = category_kpi_summary.sort_values("revenue", ascending=False).round(4)
category_kpi_summary

## 11. Monthly Revenue

In [ ]:
monthly_revenue = (
    audit_df.assign(order_month=audit_df["Order_Date"].dt.to_period("M").astype(str))
    .groupby("order_month", dropna=False)
    .agg(
        orders=("Product", "count"),
        revenue=("Sales", "sum"),
        profit=("Profit", "sum"),
        customers=("Customer_Id", "nunique"),
    )
    .reset_index()
    .sort_values("order_month")
)

monthly_revenue["profit_margin"] = monthly_revenue["profit"] / monthly_revenue["revenue"]
monthly_revenue["revenue_mom_growth"] = monthly_revenue["revenue"].pct_change()
monthly_revenue = monthly_revenue.round(4)

monthly_revenue

## 12. Export Audit Tables

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing_values.to_csv(OUTPUT_DIR / "missing_values.csv", index=False)
numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv", index=False)
categorical_distribution.to_csv(OUTPUT_DIR / "categorical_distribution.csv", index=False)
category_kpi_summary.to_csv(OUTPUT_DIR / "category_kpi_summary.csv", index=False)
monthly_revenue.to_csv(OUTPUT_DIR / "monthly_revenue.csv", index=False)
missing_examples.to_csv(OUTPUT_DIR / "missing_value_examples.csv", index=False)

print(f"Audit tables exported to: {OUTPUT_DIR}")

## 13. Key Findings

- Dataset có 51,290 rows và 16 columns.
- Date range bao phủ gần trọn năm 2018.
- Không có duplicate rows.
- Missing values rất ít, nhưng nằm ở các cột business-critical như `Sales`, `Quantity`, `Discount`, `Shipping_Cost`.
- `Payment_method` có giá trị `not_defined`, nên cần chuẩn hóa trong Phase 2.
- `Fashion` là category lớn nhất theo revenue và profit.

Decision cho Phase 2: cần chuẩn hóa column names, xử lý missing values, chuẩn hóa categorical labels, và tạo thêm date/business features.